# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [1]:
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 120.6 MB/s eta 0:00:00


# 1. Authentication

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [3]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- Markers ----
    prediction_marker: str = "<|prediction|>"
    gpt_prediction_marker: str = "<|GPT_prediction|>"

    # ---- Paths ----
    adapter_path: str = '/kaggle/input/notebooks/zygong1994/ft-0402-gpt-entropy-regularity/outputs/final_adapter'
    tokenizer_path: str = '/kaggle/input/notebooks/zygong1994/ft-0402-gpt-entropy-regularity/outputs/final_adapter'

cfg = ExperimentConfig()
print(f"Config: {cfg}")

Config: ExperimentConfig(model_name='Qwen/Qwen3-4B', max_seq_length=2048, load_in_4bit=True, bnb_4bit_quant_type='nf4', use_double_quant=True, prediction_marker='<|prediction|>', gpt_prediction_marker='<|GPT_prediction|>', adapter_path='/kaggle/input/notebooks/zygong1994/ft-0402-gpt-entropy-regularity/outputs/final_adapter', tokenizer_path='/kaggle/input/notebooks/zygong1994/ft-0402-gpt-entropy-regularity/outputs/final_adapter')


# 4. Load Eval Data

In [4]:
import pandas as pd

eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/prepare-test-data-0402/test.csv")
gpt_prob_df = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-llm-one-shot-gpt/GPT_results.csv")
eval_data['gpt_prob'] = gpt_prob_df['gpt_probability']


# TODO: user attaches gpt_prob column to eval_data here
# eval_data["gpt_prob"] = ...
assert "gpt_prob" in eval_data.columns, "Please attach gpt_prob column to eval_data"

print(f"Eval: {len(eval_data)} samples")
eval_data.head()

Eval: 200 samples


,conversation_id,conversation,full_text,outcome,text,transcript,gpt_prob
0,saas-12-conv-4816,"[{""speaker"": ""customer"", ""message"": ""Hey, I sa...","Hey, I saw your post about SmartLLM's integrat...",1,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, I saw your ...",0.60
1,saas-1-conv-4852,"[{""speaker"": ""customer"", ""message"": ""Hey, so, ...","Hey, so, I've been looking into some pricing t...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, so, I've be...",0.40
2,saas-0-conv-1824,"[{""speaker"": ""customer"", ""message"": ""Hey, than...","Hey, thanks for hopping on this call! So, um, ...",1,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, thanks for ...",0.60
3,saas-10-conv-3173,"[{""speaker"": ""sales_rep"", ""message"": ""Hey Jess...","Hey Jessica, hope you're doing well! I wanted ...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|agent|>Hey Jessica, hope y...",0.20
4,saas-18-conv-466,"[{""speaker"": ""customer"", ""message"": ""Hey, so I...","Hey, so I've been, like, struggling with track...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, so I've bee...",0.25


# 5. Load Model + Tokenizer with LoRA Adapter

In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer from adapter ----
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer vocab size: {len(tokenizer)}")

# ---- Load base model (no resize needed — no special tokens were added) ----
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)

# ---- Load LoRA adapter ----
model = PeftModel.from_pretrained(base_model, cfg.adapter_path)
model.eval()

# ---- Get label token IDs ----
token_0 = tokenizer.encode("0", add_special_tokens=False)[0]
token_1 = tokenizer.encode("1", add_special_tokens=False)[0]

print(f"Model loaded with LoRA adapter from {cfg.adapter_path}")
print(f"Token IDs -> '0': {token_0}, '1': {token_1}")

Tokenizer vocab size: 151669


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded with LoRA adapter from /kaggle/input/notebooks/zygong1994/ft-0402-gpt-entropy-regularity/outputs/final_adapter
Token IDs -> '0': 15, '1': 16


# 6. xVal Inference Helpers + Inference Functions
The key difference from standard inference: we must scale the `<|GPT_prediction|>`
token embeddings by `gpt_prob` before running the forward pass. This means we
work with `inputs_embeds` instead of `input_ids`.

In [6]:
SYSTEM_PROMPT = (
    "You are an expert B2B SaaS sales analyst specializing in conversion prediction.\n\n"
    "Given a sales conversation between a prospective customer and a sales representative, "
    "predict whether this prospect will convert (purchase a subscription or sign up for a paid plan).\n\n"
    "When assessing conversion likelihood, consider these signals:\n\n"
    "Positive signals (increase probability):\n"
    "- Explicit pain points that map to the product's value proposition\n"
    "- Questions about pricing, implementation, or timelines (buying signals)\n"
    "- Engagement depth: asking follow-ups, requesting demos or trials\n"
    "- Stakeholder involvement or mentions of internal buy-in\n"
    "- Urgency language or references to active evaluation timelines\n\n"
    "Negative signals (decrease probability):\n"
    "- Vague or noncommittal language (\"I'll think about it\", \"maybe later\")\n"
    "- Unresolved objections (cost, complexity, risk) the rep fails to address\n"
    "- Deferring to others without commitment (\"need to check with my team\")\n"
    "- Lack of engagement or short, dismissive responses\n"
    "- No clear next step established by end of conversation\n"
)


def find_marker_token_range(text, marker_str, encoding_offsets):
    """Map a marker string's character span to token indices using offset_mapping."""
    char_start = text.find(marker_str)
    if char_start == -1:
        return -1, -1
    char_end = char_start + len(marker_str)

    tok_start, tok_end = -1, -1
    for i, (cs, ce) in enumerate(encoding_offsets):
        if cs == ce == 0 and i > 0:
            continue
        if tok_start == -1 and ce > char_start:
            tok_start = i
        if cs < char_end:
            tok_end = i + 1

    return tok_start, tok_end


def build_xval_embeds(model, tokenizer, prompt, gpt_prob):
    """Tokenize prompt, scale <|GPT_prediction|> embeddings by gpt_prob, return inputs_embeds + attention_mask."""
    encoding = tokenizer(
        prompt, return_tensors="pt", return_offsets_mapping=True
    )
    input_ids = encoding["input_ids"].to(model.device)
    attention_mask = encoding["attention_mask"].to(model.device)
    offsets = encoding["offset_mapping"][0].tolist()

    # Get embeddings
    embed_layer = model.get_input_embeddings()
    inputs_embeds = embed_layer(input_ids)

    # Find and scale GPT prediction marker
    gpt_start, gpt_end = find_marker_token_range(
        prompt, cfg.gpt_prediction_marker, offsets
    )
    if gpt_start != -1:
        scale = torch.ones(input_ids.shape[1], dtype=inputs_embeds.dtype, device=model.device)
        scale[gpt_start:gpt_end] = gpt_prob
        inputs_embeds = inputs_embeds * scale.unsqueeze(0).unsqueeze(-1)

    return inputs_embeds, attention_mask, input_ids


def build_inference_prompt(system_prompt: str, transcript: str) -> str:
    return (
        f"<|system|>\n{system_prompt}\n<|/system|>\n"
        f"{transcript}\n"
        f"<|GPT_prediction|><|prediction|>"
    )


def predict_with_logit_probability(model, tokenizer, transcript: str, gpt_prob: float):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs_embeds, attention_mask, _ = build_xval_embeds(model, tokenizer, prompt, gpt_prob)

    with torch.no_grad():
        outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)

    logits = outputs.logits[:, -1, :]

    binary_logits = logits[:, [token_0, token_1]]
    probs = F.softmax(binary_logits, dim=-1)
    conversion_prob = probs[0, 1].item()

    return {
        "conversion_probability": conversion_prob,
        "predicted_class": 1 if conversion_prob > 0.5 else 0,
        "raw_logits": {"token_0": logits[0, token_0].item(), "token_1": logits[0, token_1].item()},
    }


def predict_generate(model, tokenizer, transcript: str, gpt_prob: float, max_new_tokens=10):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs_embeds, attention_mask, _ = build_xval_embeds(model, tokenizer, prompt, gpt_prob)

    with torch.no_grad():
        outputs = model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    # generate() returns full sequence; first token(s) may be from the prompt
    # when using inputs_embeds, outputs start from position 0 of the generated part
    new_tokens = tokenizer.decode(outputs[0], skip_special_tokens=False)
    return {"generated_output": new_tokens}

# 7. Test Inference on One Example

In [7]:
test_row = eval_data.iloc[0]
test_transcript = test_row["transcript"]
test_label = test_row["outcome"]
test_gpt_prob = test_row["gpt_prob"]

print("=" * 60)
print("TEST INFERENCE: Full Generation (xVal)")
print("=" * 60)
result = predict_generate(model, tokenizer, test_transcript, test_gpt_prob)
print(f"Ground truth: {test_label}")
print(f"gpt_prob: {test_gpt_prob}")
print(f"Generated output:\n{result['generated_output']}")

print("\n" + "=" * 60)
print("TEST INFERENCE: Logit-Based Probability (xVal)")
print("=" * 60)
prob_result = predict_with_logit_probability(model, tokenizer, test_transcript, test_gpt_prob)
print(f"Ground truth: {test_label}")
print(f"gpt_prob: {test_gpt_prob}")
print(f"Conversion probability: {prob_result['conversion_probability']:.4f}")
print(f"Predicted class: {prob_result['predicted_class']}")
print(f"Raw logits: {prob_result['raw_logits']}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TEST INFERENCE: Full Generation (xVal)
Ground truth: 1
gpt_prob: 0.6
Generated output:
1<|/prediction|>
<|/

TEST INFERENCE: Logit-Based Probability (xVal)
Ground truth: 1
gpt_prob: 0.6
Conversion probability: 0.5938
Predicted class: 1
Raw logits: {'token_0': 24.125, 'token_1': 24.5}


# 8. Evaluate on Full Eval Set

In [8]:
from sklearn.metrics import roc_auc_score, classification_report

eval_results = []

print("Running evaluation...")
for i in range(len(eval_data)):
    row = eval_data.iloc[i]
    prob_result = predict_with_logit_probability(
        model, tokenizer, row["transcript"], row["gpt_prob"]
    )
    eval_results.append({
        "ground_truth": row["outcome"],
        "predicted_class": prob_result["predicted_class"],
        "probability": prob_result["conversion_probability"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(eval_data)} samples")

# ---- Compute metrics ----
correct = sum(1 for r in eval_results if r["ground_truth"] == r["predicted_class"])
total = len(eval_results)
accuracy = correct / total

y_true = [r["ground_truth"] for r in eval_results]
y_pred = [r["predicted_class"] for r in eval_results]
y_prob = [r["probability"] for r in eval_results]

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Accuracy: {correct}/{total} = {accuracy:.2%}")
print(f"AUC-ROC: {roc_auc_score(y_true, y_prob):.4f}")
print(f"\n{classification_report(y_true, y_pred, target_names=['No Convert (0)', 'Convert (1)'])}")

Running evaluation...
  Processed 10/200 samples
  Processed 20/200 samples
  Processed 30/200 samples
  Processed 40/200 samples
  Processed 50/200 samples
  Processed 60/200 samples
  Processed 70/200 samples
  Processed 80/200 samples
  Processed 90/200 samples
  Processed 100/200 samples
  Processed 110/200 samples
  Processed 120/200 samples
  Processed 130/200 samples
  Processed 140/200 samples
  Processed 150/200 samples
  Processed 160/200 samples
  Processed 170/200 samples
  Processed 180/200 samples
  Processed 190/200 samples
  Processed 200/200 samples

EVALUATION RESULTS
Accuracy: 157/200 = 78.50%
AUC-ROC: 0.8654

                precision    recall  f1-score   support

No Convert (0)       0.86      0.68      0.76       100
   Convert (1)       0.74      0.89      0.81       100

      accuracy                           0.79       200
     macro avg       0.80      0.79      0.78       200
  weighted avg       0.80      0.79      0.78       200



# 9. Cross-Entropy with Ground Truth (Dirac Delta)

For a Dirac delta ground-truth distribution where $q(y=k) = 1$ for the true class and $0$ otherwise, the cross-entropy simplifies to:

$$H(q, p) = -\log p(y = y_{\text{true}})$$

where $p(y=1)$ is the model's predicted conversion probability. Per-sample cross-entropy is computed and then averaged across the eval set.

In [9]:
import numpy as np

y_true_arr = np.array(y_true)
y_prob_arr = np.array(y_prob)

# Clip probabilities to avoid log(0)
eps = 1e-7
y_prob_clipped = np.clip(y_prob_arr, eps, 1.0 - eps)

# Per-sample cross-entropy: -log p(y_true)
#   If y_true == 1: CE = -log(p)
#   If y_true == 0: CE = -log(1 - p)
per_sample_ce = -(y_true_arr * np.log(y_prob_clipped) + (1 - y_true_arr) * np.log(1 - y_prob_clipped))

mean_ce = per_sample_ce.mean()

print(f"{'='*60}")
print(f"CROSS-ENTROPY (Dirac Delta Ground Truth)")
print(f"{'='*60}")
print(f"Mean cross-entropy: {mean_ce:.4f}")
print(f"Min  cross-entropy: {per_sample_ce.min():.4f}")
print(f"Max  cross-entropy: {per_sample_ce.max():.4f}")
print(f"Std  cross-entropy: {per_sample_ce.std():.4f}")

# ---- Per-class breakdown ----
ce_class_0 = per_sample_ce[y_true_arr == 0]
ce_class_1 = per_sample_ce[y_true_arr == 1]

print(f"\nPer-class mean cross-entropy:")
print(f"  No Convert (0): {ce_class_0.mean():.4f} (n={len(ce_class_0)})")
print(f"  Convert    (1): {ce_class_1.mean():.4f} (n={len(ce_class_1)})")

CROSS-ENTROPY (Dirac Delta Ground Truth)
Mean cross-entropy: 0.4855
Min  cross-entropy: 0.0197
Max  cross-entropy: 3.0603
Std  cross-entropy: 0.6177

Per-class mean cross-entropy:
  No Convert (0): 0.6672 (n=100)
  Convert    (1): 0.3038 (n=100)


In [10]:
y_true

[np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.in

In [11]:
y_prob

[0.59375,
 0.5625,
 0.92578125,
 0.09521484375,
 0.09521484375,
 0.408203125,
 0.0849609375,
 0.87890625,
 0.09521484375,
 0.89453125,
 0.22265625,
 0.2451171875,
 0.8359375,
 0.8671875,
 0.9140625,
 0.2021484375,
 0.1640625,
 0.75390625,
 0.09521484375,
 0.8359375,
 0.294921875,
 0.1328125,
 0.294921875,
 0.1328125,
 0.8515625,
 0.10693359375,
 0.2021484375,
 0.73046875,
 0.9609375,
 0.59375,
 0.8359375,
 0.5,
 0.59375,
 0.8671875,
 0.87890625,
 0.59375,
 0.59375,
 0.796875,
 0.8515625,
 0.73046875,
 0.1328125,
 0.9453125,
 0.62109375,
 0.1484375,
 0.94140625,
 0.9140625,
 0.9609375,
 0.376953125,
 0.1484375,
 0.1328125,
 0.5625,
 0.9140625,
 0.1328125,
 0.796875,
 0.8671875,
 0.796875,
 0.8515625,
 0.93359375,
 0.62109375,
 0.8515625,
 0.87890625,
 0.9453125,
 0.10693359375,
 0.65234375,
 0.65234375,
 0.89453125,
 0.93359375,
 0.95703125,
 0.8671875,
 0.89453125,
 0.77734375,
 0.95703125,
 0.89453125,
 0.1826171875,
 0.1640625,
 0.9140625,
 0.408203125,
 0.53125,
 0.9140625,
 0.11914

# 10. Inference Template

How to use the trained xVal model for inference on new data.

## Key concept
During training, `<|GPT_prediction|>` token embeddings were scaled by `gpt_prob`.
At inference time, you must do the same — you can't just pass raw text to `model.generate()`.
Instead, use `build_xval_embeds()` to get scaled `inputs_embeds` and pass those to the model.

## Two inference modes

| Mode | Use case | Function |
|------|----------|----------|
| **Logit probability** | Get calibrated P(convert), best for metrics/thresholding | `predict_with_logit_probability()` |
| **Generation** | See full model output (e.g. "1\<\|/prediction\|\>") | `predict_generate()` |

## Step-by-step

```python
# 1. Load model + adapter (same as Section 5 above)
# 2. For each sample, you need: transcript (str) and gpt_prob (float)
# 3. Call inference:

result = predict_with_logit_probability(
    model, tokenizer,
    transcript="<|conversation|>\n<|customer|>Hi...\n<|/conversation|>",
    gpt_prob=0.73  # GPT's predicted conversion probability
)
print(result["conversion_probability"])  # e.g. 0.85
print(result["predicted_class"])         # 1
```

## What happens under the hood

```
transcript + system prompt
    ↓ build_inference_prompt()
"<|system|>...<|/system|>\n<|conversation|>...<|/conversation|>\n<|GPT_prediction|><|prediction|>"
    ↓ build_xval_embeds(model, tokenizer, prompt, gpt_prob)
    1. Tokenize → input_ids
    2. embed_tokens(input_ids) → embeddings [1, seq_len, hidden_dim]
    3. Find <|GPT_prediction|> token positions via character offsets
    4. Multiply those embeddings by gpt_prob
    ↓
model(inputs_embeds=scaled_embeds) → logits
    ↓
softmax(logits[-1, [token_0, token_1]]) → P(convert)
```

## Important notes
- **gpt_prob=0.0**: GPT marker embeddings become zero vectors (no GPT signal)
- **gpt_prob=1.0**: GPT marker embeddings unchanged (maximum GPT signal)
- The model was trained with this scaling, so you **must** provide gpt_prob at inference
- If you don't have a GPT prediction, use `gpt_prob=0.5` as a neutral baseline

In [12]:
# ---- Standalone inference example ----
# Copy this block to use the model in a different notebook/script

sample_transcript = eval_data.iloc[0]["transcript"]
sample_gpt_prob = eval_data.iloc[0]["gpt_prob"]
sample_label = eval_data.iloc[0]["outcome"]

# Method 1: Logit probability (recommended for production)
result = predict_with_logit_probability(model, tokenizer, sample_transcript, sample_gpt_prob)
print(f"Ground truth:  {sample_label}")
print(f"gpt_prob:      {sample_gpt_prob}")
print(f"P(convert):    {result['conversion_probability']:.4f}")
print(f"Prediction:    {result['predicted_class']}")

print()

# Method 2: Generation (useful for debugging)
gen_result = predict_generate(model, tokenizer, sample_transcript, sample_gpt_prob)
print(f"Generated:     {gen_result['generated_output']}")

Ground truth:  1
gpt_prob:      0.6
P(convert):    0.5938
Prediction:    1

Generated:     1<|/prediction|>
<|/
